# UR3e Real-Hardware Bring-Up

Run cells one at a time. Keep the workspace clear, keep the e-stop reachable, and keep the robot speed slider low. Motion cells are gated by `ACCEPT_RISK = False`; change it only for the one cell you intend to execute.

All robot interaction goes through `scripts/ur3e/safe_bringup.py` so the same checks are used from the terminal and the notebook.

In [ ]:
from pathlib import Path
import subprocess

ROBOT_IP = "147.175.108.138"
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "ur3e" / "safe_bringup.py").exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPT = REPO_ROOT / "scripts" / "ur3e" / "safe_bringup.py"
assert SCRIPT.exists(), SCRIPT
BASE = ["python", str(SCRIPT), "--robot-ip", ROBOT_IP]

def run_safe(*args):
    command = [*BASE, *map(str, args)]
    print("$", " ".join(command))
    return subprocess.run(command, check=False, cwd=REPO_ROOT)

def require_accept_risk(flag):
    if not flag:
        raise RuntimeError("Set ACCEPT_RISK = True in this cell only after checking the robot.")

## 1. Read-Only Status

Use this before and after every motion test. It should report `RUNNING`, `NORMAL`, no stops, and near-zero joint/TCP speed.

In [ ]:
run_safe("status")

## 2. Read-Only Watch

This confirms the arm is stationary over multiple samples.

In [ ]:
run_safe("watch", "--samples", 10, "--interval", 0.25)

## 3. Current-Position Hold Plan

This prints the hold plan but sends no control command.

In [ ]:
run_safe("hold", "--seconds", 2)

## 4. Current-Position Hold Execute

This should not intentionally move the robot. It verifies the RTDE control path.

In [ ]:
ACCEPT_RISK = False
require_accept_risk(ACCEPT_RISK)
run_safe("hold", "--seconds", 2, "--execute", "--accept-risk")

## 5. Tiny Visible Wrist Motion

`joint-index 5` is `wrist_3_joint`. Plan first, execute only when the target looks sane.

In [ ]:
run_safe("tiny-movej", "--joint-index", 5, "--delta-rad", 0.03, "--max-delta-rad", 0.04)

In [ ]:
ACCEPT_RISK = False
require_accept_risk(ACCEPT_RISK)
run_safe("tiny-movej", "--joint-index", 5, "--delta-rad", 0.03, "--max-delta-rad", 0.04, "--execute", "--accept-risk")

Move the wrist back after the visible test.

In [ ]:
ACCEPT_RISK = False
require_accept_risk(ACCEPT_RISK)
run_safe("tiny-movej", "--joint-index", 5, "--delta-rad", -0.03, "--max-delta-rad", 0.04, "--execute", "--accept-risk")

## 6. Home Plan

This is read-only. It prints the sim home target, current joint deltas, and estimated number of bounded steps.

In [ ]:
run_safe("home-plan", "--max-step-rad", 0.05)

## 7. One Bounded Home Step

This moves only one chunk toward home, limited by `--max-step-rad` per joint. Run `home-plan` again after each step.

In [ ]:
ACCEPT_RISK = False
require_accept_risk(ACCEPT_RISK)
run_safe("home-step", "--max-step-rad", 0.05, "--execute", "--accept-risk")

## 8. Policy Preview Only

This loads the learned policy, prints the guarded one-step action, and sends no control command.

In [ ]:
run_safe("policy-preview")

## 9. Policy Debug Only

This prints the raw 25D observation, normalized observation, and policy output for multiple quaternion sign conventions. It sends no control command.

In [ ]:
run_safe("policy-debug", "--full")

## 10. Single Guarded Policy Step

Do not use this until `policy-preview` and `policy-debug` look sensible. This executes exactly one small, clipped policy step.

In [ ]:
ACCEPT_RISK = False
require_accept_risk(ACCEPT_RISK)
run_safe("policy-step", "--target-offset", 0, 0, 0.005, "--execute", "--accept-risk")